# 03 — Look before you model

*Module 00 · Getting Started — notebook 3 of 11*

Before we build anything, we do what every careful practitioner does first: we look at the data. It is a quick habit that saves a great deal of grief later.

**Prerequisites:** 01 (what is machine learning?), 02 (features and the feature space).

**What you'll be able to do**
- Check the **class balance** of a dataset, and say why it matters before trusting any accuracy.
- Read a **histogram** of a feature — and a per-class histogram, to see which feature separates the classes.
- Compare features' **ranges and scales**, and connect that back to distance.
- Re-read the feature-space scatter knowing what to look for.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course import colors, datasets, viz

np.random.seed(0)
viz.use_course_style()

df = datasets.load_penguins()
X, y = datasets.penguins_xy()
FEATURES = datasets.PENGUINS_FEATURES
SPECIES_ORDER = ["Adelie", "Gentoo"]

## Where we are

Notebooks 01 and 02 gave us the vocabulary and the geometry: data as a feature matrix `X` and a label vector `y`, each example a point, and two tools — the mean and the distance.

Before we put any of that to work in a model, we pause and **look at the data**. This is the real first step of every honest machine-learning workflow — not a formality, but the moment we catch the surprises that would otherwise quietly break a model.

## Why look first

Data has a way of hiding things that hurt models:

- **Imbalance** — one class far more common than another.
- **Different scales** — one feature ranging in the thousands, another between 0 and 1.
- **Outliers and mistakes** — a flipper length of 2000 mm, or a missing value coded as -999.

A model inherits whatever is in the data — "garbage in, garbage out." So we ask three plain questions before modelling: *how many of each class do we have? what does each feature look like? are the features on comparable scales?* This notebook answers all three for our penguins.

## Question 1 — how many of each class?

The **class balance** is how the examples split across the classes. It matters more than it first appears: if 95% of penguins were Adélie, an "always say Adélie" rule would be right 95% of the time — and an accuracy of 95% would mean nothing. Knowing the balance tells us what a score has to beat to be worth anything. (We make that idea bite in notebook 06.)

In [ ]:
print(y.value_counts().to_string())

viz.plot_class_balance(y)
plt.show()

### Read the figure

We have 151 Adélie and 123 Gentoo penguins — close to balanced, with neither class dominating. That is comfortable: a useful model will have to do clearly better than the roughly 55% (151 of 274) an "always predict the majority" rule would score here. Picture instead a 95/5 split: there, beating 95% would be the real bar, and plain accuracy could flatter a useless model. Balance is the lens that tells us how to read a score.

## Question 2 — what does each feature look like?

A **histogram** bins a feature's values and counts how many examples fall in each bin — it shows the feature's **distribution**, its shape and spread, at a glance. Drawing one histogram per class, on the same axes, shows something more: whether that single feature, on its own, separates the species.

In [ ]:
viz.plot_feature_histograms(df, FEATURES, by="species")
plt.show()

### Read the figure

Each panel is one feature, with the two species drawn as overlapping histograms.

Flipper length (right) splits the species well: the two humps overlap only a little, so flipper length alone would separate most penguins. Bill length (left) is weaker — the humps overlap more, so bill length alone would leave more penguins ambiguous. Neither feature is perfect by itself, which is why we use both together: in the 2-D scatter the species separate better than along either axis alone.

## Question 3 — are the features on comparable scales?

Two features measured in the same unit can still live on very different **scales**. The `describe()` method gives, for each feature in one table: the count, the mean, the **standard deviation** (`std` — how far values typically sit from the mean, a measure of spread), and the min/max range.

In [ ]:
df[FEATURES].describe()

### Read the output

Bill length runs from about 32 to 60 mm; flipper length from about 172 to 231 mm. Both are millimetres, yet the flipper values are far larger and span a wider range. Recall the caveat from notebook 02: Euclidean distance adds up differences along each axis, so the wider-ranging flipper differences will tend to dominate any distance we compute — without our ever having decided that flipper length matters more. We only *note* this here; notebook 11 addresses it by putting features on a common scale.

Notice too what is *absent*: the counts are full (274 each), and the min/max values are all plausible penguin measurements — no missing entries, no impossible numbers. That clean bill of health is itself an EDA finding, not a step we skipped.

## Putting it together — read the scatter again

Now look once more at the feature space, holding the three answers in mind: roughly balanced classes, flipper length separating better than bill length, and two features on different scales. Where do you expect a simple rule to do well, and where to struggle?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for species, color in zip(SPECIES_ORDER, colors.CLASS_CYCLE):
    sub = X[y == species]
    ax.scatter(
        sub[FEATURES[0]],
        sub[FEATURES[1]],
        color=color,
        edgecolor=colors.COLORS["text"],
        linewidth=0.5,
        s=40,
        label=species,
        alpha=0.8,
    )

ax.set_xlabel(FEATURES[0])
ax.set_ylabel(FEATURES[1])
ax.set_title("The two species in feature space")
ax.legend(title="species")
plt.show()

### Read the figure

The species occupy two clear regions, separated mainly along the vertical (flipper) axis — as the histograms predicted. The two groups draw close along a thin band near the middle, where a few penguins of each species sit among the other's neighbours. A simple rule will do well in the two clear regions and struggle only in that narrow border band. That is a friendly problem to start on — and we still have no model: we have only learned what the data looks like, which is exactly the point.

## The words we'll keep using

Added this notebook:

- **EDA (exploratory data analysis)** — looking at data before modelling it.
- **Distribution** — how a feature's values are spread out.
- **Histogram** — a bar chart of how many values fall in each bin.
- **Class balance / imbalance** — how examples split across classes.
- **Range** — the span from a feature's minimum to its maximum.
- **Scale** — the typical size of a feature's values.
- **Outlier** — a value far from the rest, worth checking before trusting it.

## Your turn

1. From the class counts, is this dataset balanced? If instead it were 95% Adélie and 5% Gentoo, what accuracy would a model that always predicts "Adélie" get?
2. From the histograms, which single feature — bill length or flipper length — separates the two species better? How can you tell?
3. From `describe()`, which feature has the larger spread (std)? Recalling notebook 02, why does that matter for a distance that uses both features?

Jot your answers in a new cell. Notebook 04 starts using this data to train and test — so it pays to know it first.

## What you built

You've picked up the habit that opens every honest workflow — looking before modelling — and two reusable tools to do it with:

- You can read a dataset's **class balance** and say what score a model must beat.
- You can read **histograms** (overall and per class) to judge which features carry signal.
- You can compare features' **ranges and scales**, and you know why that matters for distance.
- You've re-read the scatter with informed eyes and predicted where a model will struggle.

No model has seen this data yet — and that is deliberate. Notebook 04 takes the next step: splitting the data so we can tell memorizing from genuine learning.

## References

- A. Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed., O'Reilly, 2022 — chapter 2 (take a quick look at the data; visualize to gain insights).
- J. W. Tukey, *Exploratory Data Analysis*, Addison-Wesley, 1977 — the origin of the EDA mindset.

---
Previous: **02 — Features, labels, and the feature space** · Next: **04 — Generalize, don't memorize**